# PhotoMedGemma — Kaggle Validation Notebook

**Purpose:** Run Google MedGemma 4B-IT on each breast cancer PNG, extract layer-0 attention
activations (q/k/v/o_proj), and save everything needed for local photonic comparison.

## Setup checklist

1. **Kaggle secret:** Add-ons → Secrets → `HF_TOKEN` (HuggingFace token)  
2. **Model access:** Accept terms at https://huggingface.co/google/medgemma-4b-it  
3. **Accelerator:** Session → Accelerator → GPU T4 x2 (or any GPU)  
4. **Dataset:** Upload the 5 breast cancer PNGs as a Kaggle dataset named `breast-cancer-pngs`  
   - Path on Kaggle will be: `/kaggle/input/breast-cancer-pngs/*.png`  
   - The 5 files: `10025_893612858.png`, `10011_1031443799.png`, `10042_102733848.png`,
     `10042_1648588715.png`, `10042_495770405.png`

## What this produces

| File | Contents |
|------|----------|
| `activations_<stem>.json` | Per-image: MedGemma response + layer-0 activation stats |
| `input_lm_layer0_attn_*_<stem>.npy` | Per-image: raw token activations (seq_len × dim) |
| `output_lm_layer0_attn_*_<stem>.npy` | Per-image: raw projection outputs |
| `W_q_layer0.npy` etc. | Weight matrices (saved once, same across images) |
| `kaggle_comparison.json` | Combined results — feed to `analyze_kaggle_results.py` |

In [ ]:
!pip install -q pydicom transformers accelerate bitsandbytes
!pip install -q Pillow numpy scipy

In [ ]:
import os, json, glob
import numpy as np
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# ── Configuration ──────────────────────────────────────────────────────────
MODEL_ID       = 'google/medgemma-4b-it'
IMAGE_DIR      = '/kaggle/input/breast-cancer-pngs'  # Kaggle dataset path
SVD_RANK       = 64        # photonic chip SVD truncation rank
TARGET_LAYERS  = 1         # hook layer 0 only (that's what we compiled)
QUANTIZE_4BIT  = True      # True = 4-bit NF4 (~3.5GB VRAM), False = bfloat16 (~8GB)
QUERY          = 'Describe any abnormalities visible in this medical image.'
OUTPUT_DIR     = Path('/kaggle/working/photomedgemma_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── HuggingFace token ──────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token: loaded from Kaggle secrets')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    print(f'HF token: from env ({bool(HF_TOKEN)})')

# ── Find images ────────────────────────────────────────────────────────────
image_paths = sorted(glob.glob(f'{IMAGE_DIR}/*.png'))
if not image_paths:
    # Fallback: check working directory (user may have uploaded directly)
    image_paths = sorted(glob.glob('/kaggle/working/*.png'))
if not image_paths:
    print('WARNING: No PNG images found. Creating a synthetic breast phantom for testing.')
    from PIL import Image as PILImage
    rng = np.random.default_rng(0)
    arr = np.zeros((448, 448), np.float32) + 0.05
    Y, X = np.mgrid[0:448, 0:448]
    arr += np.exp(-0.5 * (((X-224)/60)**2 + ((Y-224)/70)**2)) * 0.6
    arr += 0.02 * rng.standard_normal((448, 448))
    arr = arr.clip(0, 1)
    synth_path = '/kaggle/working/synthetic_breast.png'
    PILImage.fromarray((arr * 255).astype(np.uint8), 'L').convert('RGB').save(synth_path)
    image_paths = [synth_path]

print(f'Model:        {MODEL_ID}')
print(f'Quantization: {"4-bit NF4" if QUANTIZE_4BIT else "bfloat16"}')
print(f'SVD rank:     {SVD_RANK}')
print(f'Images found: {len(image_paths)}')
for p in image_paths:
    print(f'  {p}')

## 1. Image loader

In [ ]:
from PIL import Image

def load_png_as_pil(path: str) -> Tuple[Image.Image, dict]:
    """Load a PNG (or any image) as RGB PIL image."""
    img = Image.open(path).convert('RGB')
    stem = Path(path).stem
    meta = {
        'filename':  Path(path).name,
        'stem':      stem,
        'modality':  'PNG',
        'rows':      img.height,
        'columns':   img.width,
        'patient_id': stem,
    }
    return img, meta

# Quick preview of all images
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

n = len(image_paths)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
if n == 1:
    axes = [axes]
for ax, p in zip(axes, image_paths):
    img, meta = load_png_as_pil(p)
    ax.imshow(img)
    ax.set_title(meta['stem'][:20], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'preview.png', dpi=72)
plt.close()
print(f'Preview saved. Images: {[Path(p).stem for p in image_paths]}')

## 2. Load MedGemma (once — shared across all images)

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)

if QUANTIZE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map='auto',
    )
    print('Loaded: 4-bit NF4 quantization')
else:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, token=HF_TOKEN,
        torch_dtype=torch.bfloat16,
        device_map='auto',
    )
    print('Loaded: bfloat16')

model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}  ({n_params/1e9:.2f}B)')

## 3. Extract weights (once — same weights for all images)

In [ ]:
def extract_weight(layer_idx: int, proj_name: str, from_mlp: bool = False) -> np.ndarray:
    """Extract a weight matrix as float32 numpy. Handles 4-bit dequantization."""
    lm = model.language_model.model.layers[layer_idx]
    proj = getattr(lm.mlp if from_mlp else lm.self_attn, proj_name)
    if QUANTIZE_4BIT and hasattr(proj, 'weight') and hasattr(proj.weight, 'quant_state'):
        import bitsandbytes as bnb
        W = bnb.functional.dequantize_4bit(
            proj.weight.data, proj.weight.quant_state
        ).float().cpu().numpy()
    else:
        W = proj.weight.detach().float().cpu().numpy()
    return W

W_q = extract_weight(0, 'q_proj')
W_k = extract_weight(0, 'k_proj')
W_v = extract_weight(0, 'v_proj')
W_o = extract_weight(0, 'o_proj')

print('Layer 0 attention weights:')
for name, W in [('q_proj', W_q), ('k_proj', W_k), ('v_proj', W_v), ('o_proj', W_o)]:
    print(f'  {name}: {W.shape}  norm={np.linalg.norm(W):.3f}')

# Save weights (needed by analyze_kaggle_results.py)
for name, W in [('W_q', W_q), ('W_k', W_k), ('W_v', W_v), ('W_o', W_o)]:
    np.save(OUTPUT_DIR / f'{name}_layer0.npy', W)
print('Weights saved.')

## 4. Inline Clements decomposition + photonic forward

These functions mirror `src/compiler/clements.py` exactly.
We inline them here so the Kaggle notebook has no local dependencies.

In [ ]:
def _null_params(a, b):
    if abs(b) < 1e-15:
        return 0.0, 0.0
    phi   = float(np.angle(b) - np.angle(a))
    theta = float(2.0 * np.arctan2(abs(b), abs(a)))
    return theta % (2 * np.pi), phi % (2 * np.pi)

def _mzi_matrix(theta, phi):
    return np.array([
        [np.cos(theta/2),                         1j * np.exp(-1j*phi) * np.sin(theta/2)],
        [1j * np.exp(1j*phi) * np.sin(theta/2),  np.cos(theta/2)]
    ])

def clements_decompose_rank(U, rank=64):
    """
    Decompose unitary U — only retaining MZIs in the rank×rank subspace.
    Returns (mzis, phase_screen) where mzis are (i, col, theta, phi) tuples
    with i < rank and i+1 < rank.
    """
    N = U.shape[0]
    W = U[:rank, :rank].astype(complex).copy()  # work in rank×rank subspace
    mzis = []
    for col in range(rank - 1):
        for row in range(rank - 1, col, -1):
            theta, phi = _null_params(W[row-1, col], W[row, col])
            Td = _mzi_matrix(theta, phi).conj().T
            rows = W[[row-1, row], :].copy()
            W[[row-1, row], :] = Td @ rows
            mzis.append((row-1, col, theta, phi))
    phase_screen = np.angle(np.diag(W))
    return mzis, phase_screen

def photonic_simulate_rank(mzis, phase_screen, x):
    """
    Simulate Clements mesh on input x (rank-dimensional).
    Returns complex output vector.
    """
    field = x.astype(complex).copy()
    field *= np.exp(1j * phase_screen)
    for (i, _col, theta, phi) in reversed(mzis):
        f_ij = field[[i, i+1]].copy()
        field[[i, i+1]] = _mzi_matrix(theta, phi) @ f_ij
    return field

# Sanity check: decompose identity, simulate, error should be ~0
_I = np.eye(8, dtype=complex)
_mzis, _ps = clements_decompose_rank(_I, rank=8)
_x = np.ones(8, dtype=complex)
_err = np.linalg.norm(photonic_simulate_rank(_mzis, _ps, _x) - _x)
print(f'Clements sanity (identity): error = {_err:.2e}  (should be ~0)')

## 5. Compile layer-0 weights into photonic MZI meshes (once)

In [ ]:
from scipy.linalg import svd as scipy_svd

compiled_layers = {}  # proj_name -> {mzis_U, ps_U, mzis_Vh, ps_Vh, sigma, s0}

for proj_name, W in [('q_proj', W_q), ('k_proj', W_k), ('v_proj', W_v), ('o_proj', W_o)]:
    print(f'\nCompiling {proj_name} {W.shape}...')
    U_sv, s, Vh_sv = scipy_svd(W.astype(np.float64), full_matrices=True)
    r = min(SVD_RANK, len(s))
    energy = (s[:r]**2).sum() / (s**2).sum()
    sigma_n = s[:r] / s[0]  # normalized for optical attenuators

    print(f'  SVD: rank={r}, energy={energy:.4f}')
    print(f'  Decomposing U ({U_sv.shape[0]}x{U_sv.shape[0]}, rank-{r} subspace)...')
    mzis_U, ps_U = clements_decompose_rank(U_sv, rank=r)
    print(f'  Decomposing Vh ({Vh_sv.shape[0]}x{Vh_sv.shape[0]}, rank-{r} subspace)...')
    mzis_Vh, ps_Vh = clements_decompose_rank(Vh_sv, rank=r)
    print(f'  MZIs: U={len(mzis_U):,}  Vh={len(mzis_Vh):,}')

    compiled_layers[proj_name] = {
        'mzis_U': mzis_U, 'ps_U': ps_U,
        'mzis_Vh': mzis_Vh, 'ps_Vh': ps_Vh,
        'sigma_n': sigma_n, 's0': s[0],
        'rank': r, 'energy': float(energy),
        'n_mzis_U': len(mzis_U), 'n_mzis_Vh': len(mzis_Vh),
        'W_shape': list(W.shape),
    }

print('\nAll 4 projections compiled.')

## 6. Process each breast cancer PNG through MedGemma + photonic simulation

In [ ]:
def run_single_image(image_path: str) -> dict:
    """
    Full pipeline for one image:
      1. Load PNG
      2. Register hooks on layer-0 attention projections
      3. Run MedGemma forward (generation)
      4. Save activations to .npy
      5. Compare photonic simulation vs. digital on captured activations
      6. Return results dict
    """
    pil_img, meta = load_png_as_pil(image_path)
    stem = meta['stem']
    print(f'\n{"="*60}')
    print(f'Image: {meta["filename"]}  ({meta["rows"]}x{meta["columns"]})')

    # ── Hook registration ────────────────────────────────────────────────
    captured = {}
    hooks = []

    def make_hook(name):
        def fn(module, inp, out):
            captured[name] = {
                'input':  inp[0].detach().float().cpu().numpy(),
                'output': out.detach().float().cpu().numpy(),
            }
        return fn

    lm_layer0 = model.language_model.model.layers[0]
    for pname in ('q_proj', 'k_proj', 'v_proj', 'o_proj'):
        proj = getattr(lm_layer0.self_attn, pname)
        key  = f'lm.layer0.attn.{pname}'
        hooks.append(proj.register_forward_hook(make_hook(key)))

    # ── Forward pass ─────────────────────────────────────────────────────
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': pil_img},
        {'type': 'text',  'text':  QUERY},
    ]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors='pt',
    ).to(model.device)

    print(f'Input shape: {tuple(inputs["input_ids"].shape)}')
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False)

    for h in hooks:
        h.remove()

    response = processor.decode(
        output_ids[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )
    print(f'Response: {response[:120]}...')
    print(f'Captured {len(captured)} activations')

    # ── Save .npy activations ────────────────────────────────────────────
    activation_info = []
    for key, data in captured.items():
        inp_arr, out_arr = data['input'][0], data['output'][0]  # remove batch dim
        safe_key = key.replace('.', '_')
        np.save(OUTPUT_DIR / f'input_{safe_key}_{stem}.npy',  inp_arr)
        np.save(OUTPUT_DIR / f'output_{safe_key}_{stem}.npy', out_arr)
        activation_info.append({
            'key': key,
            'input_shape':  list(inp_arr.shape),
            'output_shape': list(out_arr.shape),
        })

    # ── Photonic comparison on captured activations ───────────────────────
    layer_comparisons = {}

    for proj_name in ('q_proj', 'k_proj', 'v_proj', 'o_proj'):
        hook_key = f'lm.layer0.attn.{proj_name}'
        if hook_key not in captured:
            continue

        act_inp = captured[hook_key]['input'][0]  # [seq_len, hidden_dim]
        comp    = compiled_layers[proj_name]
        W       = {'q_proj': W_q, 'k_proj': W_k, 'v_proj': W_v, 'o_proj': W_o}[proj_name]
        r       = comp['rank']
        W64     = W.astype(np.float64)

        n_test = min(32, act_inp.shape[0])
        errors_svd_vs_dig  = []
        errors_phot_vs_svd = []
        errors_phot_vs_dig = []

        for tok_idx in range(n_test):
            x = act_inp[tok_idx].astype(np.float64)

            # Digital full-rank
            y_dig = W64 @ x

            # SVD approximation
            x_rank = x[:W64.shape[1]]
            y_svd = (
                W_q.astype(np.float64)[:, :r] *
                comp['sigma_n'] * comp['s0']
            ) @ (W_q.astype(np.float64)[:r, :W64.shape[1]] @ x_rank)
            # Correct SVD for proper W
            from scipy.linalg import svd as _svd
            U_sv2, s2, Vh_sv2 = _svd(W64, full_matrices=False)
            y_svd = (U_sv2[:, :r] * s2[:r]) @ (Vh_sv2[:r, :] @ x)

            # Photonic simulation
            x_r = x[:min(len(x), r)].astype(complex)
            if len(x_r) < r:
                x_pad = np.zeros(r, dtype=complex)
                x_pad[:len(x_r)] = x_r
                x_r = x_pad

            after_Vh = photonic_simulate_rank(comp['mzis_Vh'], comp['ps_Vh'], x_r)
            after_sig = after_Vh * comp['sigma_n'] * comp['s0']

            y_full_u = np.zeros(W.shape[0], dtype=complex)
            # U mesh operates in rank subspace → output is rank-dim, embed back
            y_u_r = photonic_simulate_rank(comp['mzis_U'], comp['ps_U'], after_sig)
            y_full_u[:r] = y_u_r
            y_phot = y_full_u.real

            nd = np.linalg.norm(y_dig) + 1e-15
            ns = np.linalg.norm(y_svd) + 1e-15
            errors_svd_vs_dig.append(float(np.linalg.norm(y_svd  - y_dig) / nd))
            errors_phot_vs_svd.append(float(np.linalg.norm(y_phot[:r] - y_svd[:r]) / ns))
            errors_phot_vs_dig.append(float(np.linalg.norm(y_phot - y_dig) / nd))

        layer_comparisons[proj_name] = {
            'n_tokens_tested':      n_test,
            'rank':                 r,
            'energy':               comp['energy'],
            'n_mzis_total':         comp['n_mzis_U'] + comp['n_mzis_Vh'],
            'mean_svd_vs_digital':  float(np.mean(errors_svd_vs_dig)),
            'mean_photonic_vs_svd': float(np.mean(errors_phot_vs_svd)),
            'mean_photonic_vs_dig': float(np.mean(errors_phot_vs_dig)),
            'pass_photonic':        bool(np.mean(errors_phot_vs_svd) < 0.01),
        }
        status = 'PASS' if layer_comparisons[proj_name]['pass_photonic'] else 'FAIL'
        print(
            f'  {proj_name}: svd_err={np.mean(errors_svd_vs_dig):.2e} '
            f'phot_err={np.mean(errors_phot_vs_svd):.2e}  [{status}]'
        )

    result = {
        'image':              meta['filename'],
        'stem':               stem,
        'image_size':         [meta['rows'], meta['columns']],
        'model_id':           MODEL_ID,
        'quantized_4bit':     QUANTIZE_4BIT,
        'query':              QUERY,
        'response':           response,
        'n_input_tokens':     int(inputs['input_ids'].shape[1]),
        'activations':        activation_info,
        'layer_comparisons':  layer_comparisons,
        'all_pass':           all(v['pass_photonic'] for v in layer_comparisons.values()),
    }

    # Save per-image JSON
    json_path = OUTPUT_DIR / f'activations_{stem}.json'
    json_path.write_text(json.dumps(result, indent=2))
    print(f'Saved: {json_path.name}')

    return result

print('run_single_image() defined.')

In [ ]:
# ── Process all images ───────────────────────────────────────────────────────
all_results = []

for img_path in image_paths:
    try:
        result = run_single_image(img_path)
        all_results.append(result)
    except Exception as e:
        print(f'ERROR processing {img_path}: {e}')
        import traceback
        traceback.print_exc()

print(f'\nProcessed {len(all_results)}/{len(image_paths)} images successfully.')

## 7. Save combined kaggle_comparison.json

This file is consumed by `scripts/analyze_kaggle_results.py` locally.

In [ ]:
# Build combined structure compatible with analyze_kaggle_results.py
# Use the first image's q_proj comparison as the top-level 'layer_0_q_proj' key

combined = {
    'model_id':    MODEL_ID,
    'quantized':   QUANTIZE_4BIT,
    'query':       QUERY,
    'svd_rank':    SVD_RANK,
    'n_images':    len(all_results),
    'images':      [r['image'] for r in all_results],
    'per_image':   all_results,
    # Legacy single-image fields (using first image, q_proj) for analyze_kaggle_results.py
    'response':    all_results[0]['response'] if all_results else '',
    'dicom_meta':  {'modality': 'PNG', 'patient_id': all_results[0]['stem']} if all_results else {},
    'layer_0_q_proj': {
        'weight_shape':         compiled_layers['q_proj']['W_shape'],
        'effective_rank':       compiled_layers['q_proj']['rank'],
        'energy_retained':      compiled_layers['q_proj']['energy'],
        'n_mzis_U':             compiled_layers['q_proj']['n_mzis_U'],
        'n_mzis_Vh':            compiled_layers['q_proj']['n_mzis_Vh'],
        'mean_svd_vs_digital':  float(np.mean([
            r['layer_comparisons']['q_proj']['mean_svd_vs_digital']
            for r in all_results if 'q_proj' in r.get('layer_comparisons', {})
        ])) if all_results else 0.0,
        'mean_photonic_vs_svd': float(np.mean([
            r['layer_comparisons']['q_proj']['mean_photonic_vs_svd']
            for r in all_results if 'q_proj' in r.get('layer_comparisons', {})
        ])) if all_results else 0.0,
        'token_comparisons': [],  # full data in per_image
    }
}

combined_path = OUTPUT_DIR / 'kaggle_comparison.json'
combined_path.write_text(json.dumps(combined, indent=2))
print(f'Saved: {combined_path}')

## 8. Final summary

In [ ]:
print('=' * 70)
print('PhotoMedGemma x MedGemma — Breast Cancer PNG Validation')
print('=' * 70)
print(f'  Model:        {MODEL_ID}')
print(f'  Quantization: {"4-bit NF4" if QUANTIZE_4BIT else "bfloat16"}')
print(f'  SVD rank:     {SVD_RANK}')
print(f'  Images:       {len(all_results)}')
print()

print(f'  {"Image":<35} {"q_phot_err":>12} {"k_phot_err":>12} {"v_phot_err":>12} {"o_phot_err":>12}  All-PASS')
print('-' * 95)
for r in all_results:
    lc = r.get('layer_comparisons', {})
    row = f"  {r['stem']:<35}"
    for pn in ('q_proj', 'k_proj', 'v_proj', 'o_proj'):
        err = lc.get(pn, {}).get('mean_photonic_vs_svd', float('nan'))
        row += f' {err:>12.3e}'
    row += f"  {'YES' if r['all_pass'] else 'NO'}"
    print(row)

print()
print('Responses:')
print('-' * 70)
for r in all_results:
    print(f"\n  [{r['stem']}]")
    for line in r['response'].split('\n')[:4]:
        print(f'  {line}')

print()
print('Output files (download from Kaggle output tab):')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name:<60} {f.stat().st_size / 1024:>8.0f} KB')

print()
print('Next step — run locally after downloading:')
print('  cp ~/Downloads/kaggle_comparison.json output/simulations/')
print('  cp ~/Downloads/W_*_layer0.npy         output/simulations/')
print('  python3 scripts/analyze_kaggle_results.py \\')
print('    --results output/simulations/kaggle_comparison.json \\')
print('    --weights-dir output/simulations/')
print('=' * 70)